# Classification
`capabilities/classification/clip_classifier.py`  
`capabilities/classification/vlm_classifier.py`

Two zero-shot classifiers — no fixed class list, no fine-tuning required:

| Class | Backend | Speed | Cost | Strengths |
|-------|---------|-------|------|----------|
| `CLIPClassifier` | CLIP ViT (local) | 1–3s CPU | Free | Fast, offline, consistent |
| `VLMClassifier` | Gemini Flash (cloud) | 1–2s | ~$0.001/call | Reasoning, multi-field, free-form |

## Setup

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from capabilities.classification.clip_classifier import CLIPClassifier
from sprout_detection.utils.image_gen import make_sprout_image, make_bare_soil_image, make_ambiguous_image

tmp = tempfile.mkdtemp()

# Create test images
paths = {}
for name, fn, kwargs in [
    ('sprout',    make_sprout_image,   {'seed': 1}),
    ('bare_soil', make_bare_soil_image, {'seed': 2}),
    ('ambiguous', make_ambiguous_image, {'seed': 3}),
]:
    p = os.path.join(tmp, f'{name}.png')
    cv2.imwrite(p, fn(**kwargs))
    paths[name] = p

print('Test images ready')

---
# Part 1 — CLIPClassifier

> Downloads ~350 MB on first use. Requires `clip` and `torch`.

## Basic usage — classify into custom labels

In [ ]:
clf = CLIPClassifier()   # lazy-loads ViT-B/32 on first call

result = clf.classify(
    paths['sprout'],
    labels=['healthy green plant', 'bare soil', 'diseased plant', 'dead plant']
)

print(f'Top label    : {result.top_label}')
print(f'Confidence   : {result.confidence:.3f}')
print(f'All scores   : ')
for label, score in sorted(result.all_scores.items(), key=lambda x: -x[1]):
    bar = '█' * int(score * 30)
    print(f'  {label:<25} {score:.3f}  {bar}')
print(f'Inference ms : {result.duration_ms:.0f}')

## Binary decision with positive_indices

In [ ]:
# Sprout vs no-sprout: indices 0 and 1 are positive, 2 and 3 are negative
labels = ['green sprout', 'seedling emerging', 'bare soil', 'empty pot']

for name, path in paths.items():
    r = clf.classify(path, labels=labels, positive_indices=[0, 1])
    decision_str = '✅ SPROUT' if r.binary_decision else '❌ NO SPROUT'
    print(f'{name:<12}: {decision_str}  pos={r.positive_score:.3f}  neg={r.negative_score:.3f}  top="{r.top_label}"')

## classify_multi_question() — answer multiple questions about one image

In [ ]:
questions = {
    'health':    ['healthy plant', 'stressed plant', 'dying plant'],
    'stage':     ['bare soil no plant', 'tiny sprout', 'small seedling', 'mature plant'],
    'coverage':  ['sparse sparse coverage', 'moderate coverage', 'dense full coverage'],
    'condition': ['good growing conditions', 'poor growing conditions'],
}

multi = clf.classify_multi_question(paths['sprout'], questions=questions)

print('Multi-question results for sprout image:')
for q_name, r in multi.items():
    print(f'  {q_name:<12}: "{r.top_label}"  conf={r.confidence:.3f}')

## Compare all three images across growth stages

In [ ]:
stage_labels = ['bare soil no plant', 'tiny sprout just emerging', 'small healthy seedling', 'large mature plant']

all_scores = {}
for name, path in paths.items():
    r = clf.classify(path, labels=stage_labels)
    all_scores[name] = [r.all_scores[l] for l in stage_labels]

# Heatmap
fig, ax = plt.subplots(figsize=(10, 4))
data = np.array(list(all_scores.values()))
im = ax.imshow(data, cmap='YlGn', vmin=0, vmax=0.8, aspect='auto')

ax.set_xticks(range(len(stage_labels)))
ax.set_xticklabels([l.replace(' ', '\n') for l in stage_labels], fontsize=8)
ax.set_yticks(range(len(all_scores)))
ax.set_yticklabels(list(all_scores.keys()), fontsize=10)

for i in range(len(all_scores)):
    for j in range(len(stage_labels)):
        ax.text(j, i, f'{data[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='black' if data[i,j] < 0.5 else 'white')

plt.colorbar(im, ax=ax, label='CLIP probability')
ax.set_title('CLIP Zero-Shot Stage Classification — Probability Heatmap', fontsize=11)
plt.tight_layout()
plt.show()

## Different CLIP model variants

In [ ]:
# ViT-B/32 (default) vs ViT-L/14 (more accurate, 2.5× larger)
# Only run if you want to compare — loads a second 900 MB model

clf_large = CLIPClassifier(model_variant='ViT-L/14')

labels_health = ['healthy plant', 'stressed plant', 'diseased plant']

r_small = clf.classify(paths['sprout'],      labels=labels_health)
r_large = clf_large.classify(paths['sprout'], labels=labels_health)

print('ViT-B/32:')
for l, s in r_small.all_scores.items(): print(f'  {l:<20}: {s:.3f}')
print()
print('ViT-L/14:')
for l, s in r_large.all_scores.items(): print(f'  {l:<20}: {s:.3f}')

---
# Part 2 — VLMClassifier

> Requires a Gemini API key. Costs ~$0.001 per call.

In [ ]:
from capabilities.classification.vlm_classifier import VLMClassifier

# Set your API key here or via environment variable
# import os; os.environ['GEMINI_API_KEY'] = 'your-key'

vlm = VLMClassifier()   # reads GEMINI_API_KEY from environment
print(repr(vlm))

## classify() — labelled classification with reasoning

In [ ]:
result = vlm.classify(
    paths['sprout'],
    question='What is the current growth stage of this plant?',
    labels=['bare soil', 'germination', 'seedling', 'vegetative', 'flowering'],
)

print(f'Label     : {result.top_label}')
print(f'Confidence: {result.confidence:.2f}')
print(f'Reasoning : {result.reasoning}')
print(f'Time      : {result.duration_ms:.0f} ms')

## analyse() — free-form prompt

In [ ]:
result = vlm.analyse(
    paths['sprout'],
    prompt='Describe the health status of this plant and any visible issues. Be concise.'
)

print(result.analysis_text)

## analyse_structured() — fill in a schema

In [ ]:
import json

result = vlm.analyse_structured(
    paths['sprout'],
    schema={
        'growth_stage':   'one of: germination, seedling, vegetative, flowering, fruiting',
        'health_score':   'float 0.0 (dead) to 1.0 (perfect health)',
        'visible_issues': 'list of strings describing problems, empty list if none',
        'recommendation': 'one actionable sentence for the grower',
        'confidence':     'float 0.0 to 1.0 — your confidence in this assessment',
    }
)

print('Structured output:')
print(json.dumps(result.structured_data, indent=2))